In [1]:
import numpy as np
import pandas as pd
import os
%matplotlib widget
import matplotlib.pyplot as plt

###
In Hypoxia_Statistics_AllFiles_part0.ipynb, we've computed all hypoxia variables for all sleeplab files.

Now, Moin has provided us with some lists of MRNs/names, let's see what we can match, fill up his list.

[note, this is version 2 of this code. previously, there was another notebook]

In [2]:
sleeplab_table = pd.read_csv(f'sleeplab_table_hypoxia.csv')
sleeplab_table.drop(['Unnamed: 0'], axis=1, inplace=True)
# table_grass = pd.read_csv('/media/mad3/Datasets_ConvertedData/sleeplab/grass_studies_list.csv')
# table_natus = pd.read_csv('/media/mad3/Datasets_ConvertedData/sleeplab/natus_studies_list.csv')
# assert np.all(table_grass.columns == table_natus.columns)
# sleeplab_table = pd.concat([table_grass, table_natus], axis=0)
# sleeplab_table = sleeplab_table[pd.notna(sleeplab_table.MRN)]
# sleeplab_table.reset_index(inplace=True, drop=True)
# print(sleeplab_table.shape)

sleeplab_table['MRN_int'] = [int(str(x).replace('-','').replace('/','').replace('X','0')) for x in sleeplab_table['MRN']]
sleeplab_table.shape

(25231, 19)

In [3]:
sleeplab_table.head()

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific,MRN_int
0,Anatasia_Lori_091611_2209.000,Z11530888,424-66-43,Anastasia,Lori E,Female,1972-08-11,2011-09-16,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.641667,0.3,0.7,0.000000,0.000000,0.000000,4246643
1,Gronewold_Catherine W_041615_2224.000,Z8592929,237-74-50,Gronewold,Catherine,Female,1948-12-13,2015-04-16,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.816667,26.1,33.7,16.903167,13.744833,3.158333,2377450
2,Morris_Martha_112512_2118.000,Z9992290,512-71-58,Morris,Martha,Female,1947-04-30,2012-11-25,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,0.883333,0.0,0.0,5.814500,4.109917,1.704583,5127158
3,Schlein_Toby_012717_2300.000,Z6488130,354-22-12,Schlein,Toby,Female,1945-11-16,2017-01-27,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.350000,8.4,6.4,6.542250,6.542250,0.000000,3542212
4,Callahan_Marian_042010_2158.000,Z10588347,160-39-24,Callahan,Marian,Female,1959-08-30,2010-04-20,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.366667,2.8,2.1,0.815583,0.199500,0.616083,1603924


In [4]:
run_fuzzy_matching = False # if False, load previously run and saved table.

if run_fuzzy_matching:
    
    hypoxic_table = pd.read_excel('Hypoxic_Study_Table.xlsx', engine='openpyxl')
    hypoxic_table = hypoxic_table.dropna(how='all', axis=1)

    hypoxic_table['LastName'] = [str(x).split(', ')[0] for x in hypoxic_table.Name.values]
    hypoxic_table['FirstName'] = [str(x).split(', ')[1] if ', ' in str(x) else np.nan for x in hypoxic_table.Name.values]

    
    import Levenshtein

    hypoxic_table['NameMatch'] = 0
    hypoxic_table['MatchedMRN'] = np.nan
    hypoxic_table['MatchedLastName'] = np.nan
    hypoxic_table['MatchedFirstName'] = np.nan
    hypoxic_table['MatchedDOB'] = np.nan

    for index, row in hypoxic_table.iterrows():
        
        try:
            if pd.isna(row.Name):
                continue

            for index2, row2 in sleeplab_table.iterrows():
    #             try:
                distance_lastname = Levenshtein.distance(str(row.LastName).lower(), str(row2.LastName).lower())
                distance_firstname = Levenshtein.distance(str(row.FirstName).lower(), str(row2.FirstName).lower())

                min_distance_lastname = 2
                min_distance_firstname = 2
                if len(row.LastName) < 3:
                    min_distance_lastname = 0
                if len(row.FirstName) < 3:
                    min_distance_firstname = 0

                if ((distance_lastname <= min_distance_lastname) & (distance_firstname <= min_distance_firstname)) | ((distance_lastname == 0) & (distance_firstname <= min_distance_firstname+1)):
                    hypoxic_table.loc[index, 'NameMatch'] = 1
                    hypoxic_table.loc[index, 'MatchedMRN'] = int(row2.MRN_int)
                    hypoxic_table.loc[index, 'MatchedLastName'] = row2.LastName
                    hypoxic_table.loc[index, 'MatchedFirstName'] = row2.FirstName
                    hypoxic_table.loc[index, 'MatchedDOB'] = row2.DateOfBirth

        except Exception as e:
            print(e)
            continue
            
    hypoxic_table.to_csv('Hypoxic_Study_Table.csv', index=False)
    
else:
    hypoxic_table = pd.read_csv('Hypoxic_Study_Table.csv')

print(hypoxic_table.shape)
hypoxic_table

(936, 48)


,File,Line,ID,Name,Multiple values,EMPI,EPIC_PMRN,MRN_Type,MRN,Report_Number,...,PAsat,SVCsat,LVsat,LastName,FirstName,NameMatch,MatchedMRN,MatchedLastName,MatchedFirstName,MatchedDOB
0,xaa,11252.0,xaa-11252,"Cunningham, William",*,100006892.0,1.002869e+10,MGH,75751.0,HA004046,...,NaN,NaN,NaN,Cunningham,William,0,NaN,NaN,NaN,NaN
1,xaa,14878.0,xaa-14878,"CANADY, MILDRED",NaN,100001724.0,1.004001e+10,BWH,371260.0,4412350964,...,70.0,71.0,NaN,CANADY,MILDRED,0,NaN,NaN,NaN,NaN
2,xaa,31930.0,xaa-31930,"Matthews, John",NaN,100080788.0,1.004335e+10,MGH,978221.0,HC014805,...,NaN,NaN,NaN,Matthews,John,1,978221.0,Matthews,John D,1955-01-04
3,xaa,35634.0,xaa-35634,"CROKE, WILLIAM",NaN,100076644.0,1.004311e+10,MGH,937117.0,SIS31526,...,NaN,NaN,NaN,CROKE,WILLIAM,1,2408320.0,Crane,William J,1956-09-24
4,xaa,39029.0,xaa-39029,"Walker, Desiree",NaN,100075764.0,1.004306e+10,BWH,3904687.0,E12638408,...,NaN,NaN,NaN,Walker,Desiree,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
931,NaN,NaN,NaN,"Vondohlen, Geralyn",NaN,100478240.0,1.003014e+10,BWH,9095431.0,NaN,...,NaN,NaN,NaN,Vondohlen,Geralyn,0,NaN,NaN,NaN,NaN
932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
933,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
934,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN


In [5]:
control_table = pd.read_excel('/scr/wolfgang/Hypoxia_Study/control_group.xls')
control_table = control_table.dropna(how='all', axis=1)
print(control_table.shape)
control_table = control_table[control_table.MRN_Type.apply(lambda x: 'MGH' in x)]
control_table['MRN_original'] = control_table['MRN']
control_table.loc[control_table.MRN_Type == 'PMRN, MGH, BWH', 'MRN'] = control_table.loc[control_table.MRN_Type == 'PMRN, MGH, BWH'].MRN.apply(lambda x: x.split(',')[1].strip())
control_table.loc[control_table.MRN_Type == 'PMRN, MGH', 'MRN'] = control_table.loc[control_table.MRN_Type == 'PMRN, MGH'].MRN.apply(lambda x: x.split(',')[1].strip())
control_table['MRN'] = control_table['MRN'].astype(int)

(19548, 16)


In [6]:
hypoxic_table.head(3)

,File,Line,ID,Name,Multiple values,EMPI,EPIC_PMRN,MRN_Type,MRN,Report_Number,...,PAsat,SVCsat,LVsat,LastName,FirstName,NameMatch,MatchedMRN,MatchedLastName,MatchedFirstName,MatchedDOB
0,xaa,11252.0,xaa-11252,"Cunningham, William",*,100006892.0,1.002869e+10,MGH,75751.0,HA004046,...,NaN,NaN,NaN,Cunningham,William,0,NaN,NaN,NaN,NaN
1,xaa,14878.0,xaa-14878,"CANADY, MILDRED",NaN,100001724.0,1.004001e+10,BWH,371260.0,4412350964,...,70.0,71.0,NaN,CANADY,MILDRED,0,NaN,NaN,NaN,NaN
2,xaa,31930.0,xaa-31930,"Matthews, John",NaN,100080788.0,1.004335e+10,MGH,978221.0,HC014805,...,NaN,NaN,NaN,Matthews,John,1,978221.0,Matthews,John D,1955-01-04


In [7]:
hypoxic_table['MRN_found'] = 0

hypoxic_table.loc[np.isin(hypoxic_table.MRN, sleeplab_table.MRN_int), 'MRN_found'] = 1
print(hypoxic_table.shape[0])
print(hypoxic_table.MRN_found.sum())
print(hypoxic_table.NameMatch.sum())
print(hypoxic_table.loc[ (hypoxic_table.MRN_found == 0) & (hypoxic_table.NameMatch == 1), :].shape[0])

MRNs_only_matched_by_name = hypoxic_table.loc[ (hypoxic_table.MRN_found == 0) & (hypoxic_table.NameMatch == 1), 'MatchedMRN'].values

936
409
524
135


In [8]:
hypoxic_table_matched_mrn = sleeplab_table.loc[np.isin(sleeplab_table.MRN_int, hypoxic_table.MRN), :]
hypoxic_table_matched_name = sleeplab_table.loc[np.isin(sleeplab_table.MRN_int, MRNs_only_matched_by_name), :]
control_table_matched_mrn = sleeplab_table.loc[np.isin(sleeplab_table.MRN_int, control_table.MRN), :]

print(hypoxic_table_matched_mrn.shape)
print(hypoxic_table_matched_name.shape)
print(control_table_matched_mrn.shape)

(677, 19)
(193, 19)
(12050, 19)


In [9]:
# sleeplab entries with hypoxia values matched by MRN with Moin's hypoxia cohort. 
hypoxic_table_matched_mrn.head(2)

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific,MRN_int
15,Deberardinis_Marion_072817_2242.000,Z6377495,162-80-27,Deberardinis,Marion,Female,1950-12-27,2017-07-28,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.208333,22.7,30.9,29.696250,29.102250,0.594000,1628027
74,Ansaldi_Laurie_121412_2300.000,Z11636199,225-96-68,Ansaldi,Laurie,Female,1961-05-30,2012-12-14,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.416667,0.9,0.3,1.072083,0.416667,0.655417,2259668


In [10]:
# sleeplab entries with hypoxia values fuzzy-matched by name, not with MRN. these entries might be correct but would maybe need to be manually be reviewed.
hypoxic_table_matched_name.head(2)

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific,MRN_int
15,Deberardinis_Marion_072817_2242.000,Z6377495,162-80-27,Deberardinis,Marion,Female,1950-12-27,2017-07-28,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.208333,22.7,30.9,29.69625,29.10225,0.594,1628027
140,Olans_Stephen A_042918_2249.000,Z6411554,236-94-67,Olans,Stephen A,Male,1957-05-20,2018-04-29,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,4.916667,8.9,4.2,0.00000,0.00000,0.000,2369467


In [11]:
# sleeplab entries with hypoxia values matched by MRN.
control_table_matched_mrn.head(2)

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific,MRN_int
1,Gronewold_Catherine W_041615_2224.000,Z8592929,237-74-50,Gronewold,Catherine,Female,1948-12-13,2015-04-16,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.816667,26.1,33.7,16.903167,13.744833,3.158333,2377450
3,Schlein_Toby_012717_2300.000,Z6488130,354-22-12,Schlein,Toby,Female,1945-11-16,2017-01-27,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.350000,8.4,6.4,6.542250,6.542250,0.000000,3542212


In [12]:
print(hypoxic_table_matched_mrn.shape)
print(hypoxic_table_matched_name.shape)
print(control_table_matched_mrn.shape)

hypoxic_table_matched_mrn = hypoxic_table_matched_mrn[pd.notna(hypoxic_table_matched_mrn.AHI)]
hypoxic_table_matched_name = hypoxic_table_matched_name[pd.notna(hypoxic_table_matched_name.AHI)]
control_table_matched_mrn = control_table_matched_mrn[pd.notna(control_table_matched_mrn.AHI)]
print('with removed entries where no annotations were available or sth. else lead no invalid AHI variables:')
print(hypoxic_table_matched_mrn.shape)
print(hypoxic_table_matched_mrn.MRN.unique().shape)
print(hypoxic_table_matched_name.shape)
print(hypoxic_table_matched_name.MRN.unique().shape)
print(control_table_matched_mrn.shape)
print(control_table_matched_mrn.MRN.unique().shape)


(677, 19)
(193, 19)
(12050, 19)
with removed entries where no annotations were available or sth. else lead no invalid AHI variables:
(603, 19)
(384,)
(175, 19)
(125,)
(10839, 19)
(7686,)


In [13]:
fig, ax = plt.subplots(2, 2)

alpha = 0.4
s = 10

ax = ax.flatten()

ax[0].scatter(hypoxic_table_matched_mrn.AHI, hypoxic_table_matched_mrn.HypoxiaBurden, marker='o', facecolor='None', edgecolor='k', alpha=alpha, s=s)
ax[0].set_xlabel('AHI')
ax[0].set_ylabel('HypoxiaBurden')

ax[1].scatter(hypoxic_table_matched_mrn.AHI, hypoxic_table_matched_mrn.T90_minutes, marker='o', facecolor='None', edgecolor='k', alpha=alpha, s=s)
ax[1].set_xlabel('AHI')
ax[1].set_ylabel('T90_minutes')

ax[2].scatter(hypoxic_table_matched_mrn.AHI, hypoxic_table_matched_mrn.T90_minutes_desat, marker='o', facecolor='None', edgecolor='k', alpha=alpha, s=s)
ax[2].set_xlabel('AHI')
ax[2].set_ylabel('T90_minutes_desat')

ax[3].scatter(hypoxic_table_matched_mrn.AHI, hypoxic_table_matched_mrn.T90_minutes_unspecific, marker='o', facecolor='None', edgecolor='k', alpha=alpha, s=s)
ax[3].set_xlabel('AHI')
ax[3].set_ylabel('T90_minutes_unspecific')

plt.tight_layout()
plt.savefig('hypoxia_indices_scatter.jpg', dpi=300)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [14]:
if 0:
    
    import sys
    sys.path.append('/home/wolfgang/repos/sleep_general')
    from mgh_sleeplab import load_mgh_signal, annotations_preprocess, vectorize_respiratory_events, vectorize_sleep_stages

    from sleep_analysis_functions import compute_spo2_clean, compute_hypoxia_burden, desaturation_detection, hypoxaemic_burden_minutes
    import datetime

    def compute_hypoxia_statistics(data, apnea, sleep_stage, fs):

        data['apnea'] = apnea

        dt_start = pd.Timestamp('2000-01-01 00:00:00')
        dt_end = dt_start + datetime.timedelta(seconds=(data.shape[0]-1) / fs)
        pseudo_dt_index = pd.date_range(start=dt_start, end=dt_end, periods=data.shape[0])
        data.index = pseudo_dt_index

        data = compute_spo2_clean(data, fs=fs)
        data['spo2'] = data['spo2_clean']

        data['apnea_binary'] = np.isin(data['apnea'],[1,2,3,4]).astype(int)
        data['apnea_end'] = np.isin(data['apnea_binary'].diff(), [-1])

        sleep_stage = sleep_stage[np.logical_not(pd.isna(sleep_stage))]
        hours_sleep = sum(sleep_stage<5)/fs/3600

        data, hypoxia_burden = compute_hypoxia_burden(data, fs, hours_sleep=hours_sleep, apnea_name = 'apnea')

        T90burden, T90desaturation, T90nonspecific = hypoxaemic_burden_minutes(data['spo2'].values, fs)

        AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)

        return hypoxia_burden, AHI, hours_sleep, T90burden, T90desaturation, T90nonspecific


In [15]:
if 0:
    
    i = 0

    path_signal = inspect.iloc[i].path_signal
    path_annotations = inspect.iloc[i].path_annotations
    print(path_signal)
    print(path_annotations)

    # read in raw data:
    signal, params = load_mgh_signal(path_signal)
    annotations = pd.read_csv(path_annotations)

    fs = int(params['Fs'])
    signal_len = signal.shape[0]

    # get respiratory event and sleep stage array:
    annotations = annotations_preprocess(annotations, fs)
    resp = vectorize_respiratory_events(annotations, signal_len)
    sleep_stage = vectorize_sleep_stages(annotations, signal_len)
    
    hypoxia_burden, ahi, total_sleep_time, T90burden, T90desaturation, T90nonspecific = compute_hypoxia_statistics(signal, resp, sleep_stage, fs)
    
    
    fig, ax = plt.subplots(4, 1, sharex=True)

    ax[0].plot(signal['ptaf'])
    ax[1].plot(signal['abd'])
    ax[2].plot(signal['spo2'])
    ax[2].set_ylim([80, 100.1])

    ax[3].plot(resp)


In [16]:
hypoxic_table_matched_mrn

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific,MRN_int
15,Deberardinis_Marion_072817_2242.000,Z6377495,162-80-27,Deberardinis,Marion,Female,1950-12-27,2017-07-28,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.208333,22.7,30.9,29.696250,29.102250,0.594000,1628027
74,Ansaldi_Laurie_121412_2300.000,Z11636199,225-96-68,Ansaldi,Laurie,Female,1961-05-30,2012-12-14,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.416667,0.9,0.3,1.072083,0.416667,0.655417,2259668
149,Amazu_Adeline_050714_2348.000,Z12907166,345-73-99,Amazu,Adeline,Female,1949-09-04,2014-05-07,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.491667,13.7,8.4,0.000000,0.000000,0.000000,3457399
179,El-Sherbini_Ahmad_060209_2306.000,Z6365231,250-52-26,Elsherbini,Ahmed A,Male,1940-09-01,2009-06-02,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.041667,8.4,15.0,7.214417,5.148417,2.066000,2505226
215,Mullen_Robert_110211_2156.000,Z8714448,265-72-98,Mullen,Robert K,Male,1931-05-25,2011-11-02,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,3.775000,9.0,7.8,3.115333,2.082333,1.033000,2657298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24474,Dube~ Daniel_54e840ea-cfe4-4290-bc61-eb6421138673,Z14350610,570-36-51,Dube,Daniel A,Male,1951-08-09,2018-11-28,Diagnostic,M:\Datasets_ConvertedData\sleeplab\natus_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,3.400000,0.0,0.0,2.818200,1.165400,1.652800,5703651
24785,Stimson~ Marga_4cbe2f23-2d28-4eb7-ac00-9318170...,Z9004628,428-90-19,Stimson,Margaret M,Female,1953-09-17,2019-02-14,Diagnostic,M:\Datasets_ConvertedData\sleeplab\natus_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,5.892422,0.0,0.0,1.405241,1.405241,0.000000,4289019
24809,Capraro~ David_1912079a-30d4-4145-9ed4-b5e8126...,Z15770002,609-18-13,Capraro,David,Male,1959-02-11,2018-11-14,ASV,M:\Datasets_ConvertedData\sleeplab\natus_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,5.350000,0.0,0.0,0.388467,0.178600,0.209867,6091813
24875,Vogel~ Michael_da9c24b2-7cde-4d3d-b60d-d68cde0...,Z8106184,100-09-32,Vogel,Michael H,Male,1951-02-01,2019-04-16,Split Night,M:\Datasets_ConvertedData\sleeplab\natus_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,/media/mad3/Datasets_ConvertedData/sleeplab/na...,5.091667,0.0,0.0,7.515104,7.515104,0.000000,1000932


In [17]:
# Prepare Table to share (w/ only selected variables)

vars_share = ['PatientID', 'MRN', 'LastName', 'FirstName', 'Sex', 'DateOfBirth', 'DateOfVisit', 'TypeOfTest', 'AHI', 'HypoxiaBurden',
       'T90_minutes', 'T90_minutes_desat', 'T90_minutes_unspecific']

hypoxic_table_matched_mrn = hypoxic_table_matched_mrn[vars_share].sort_values(by=['PatientID', 'MRN', 'DateOfVisit'])
hypoxic_table_matched_name = hypoxic_table_matched_name[vars_share].sort_values(by=['PatientID', 'MRN', 'DateOfVisit'])
control_table_matched_mrn = control_table_matched_mrn[vars_share].sort_values(by=['PatientID', 'MRN', 'DateOfVisit'])

In [18]:
hypoxic_table_matched_mrn.to_csv('entries_cohort_hypoxia_matchedMRN.csv', index=False)
hypoxic_table_matched_name.to_csv('entries_cohort_hypoxia_fuzzymatchedName.csv', index=False)
control_table_matched_mrn.to_csv('entries_cohort_control_matchedMRN.csv', index=False)